# 04 — Multi-Turn Conversations: Agents with Memory

**What you'll learn:**
- How `AgentSession` preserves conversation state across runs
- Building a multi-turn banking chatbot
- Serializing and restoring sessions

---

## The Problem

By default, each `agent.run()` call is **stateless** — the agent has no memory
of previous turns. But real conversations need context:

- Customer asks about account types → follows up about interest rates → references something from turn 1
- Without sessions, the agent would say "I don't know what account you mean" every time

**`AgentSession`** solves this. It's a conversation state container you pass across runs.

In [9]:
import os

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv

from agent_framework.azure import AzureOpenAIResponsesClient

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()

## Without a Session (Stateless)

Let's first see what happens **without** a session. The agent forgets
everything between calls.

In [10]:
agent = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    credential=credential,
).as_agent(
    name="BankingAdvisor",
    instructions=(
        "You are a helpful banking advisor at First National Bank. "
        "You help customers understand products, rates, and account options. "
        "Be friendly and professional. Keep answers concise."
    ),
)

# Turn 1
r1 = await agent.run("Hi, my name is Priya and I'm interested in opening a savings account.")
print(f"Turn 1: {r1.text}\n")

# Turn 2 — agent won't know who "I" is or what we discussed
r2 = await agent.run("What interest rate would I get?")
print(f"Turn 2: {r2.text}")

Turn 1: Hi Priya! It’s great to hear that you’re interested in opening a savings account with us. We offer several options designed to help you grow your savings. May I ask what your goals are—like earning higher interest, needing easy access to funds, or anything specific—so I can guide you to the best account?

Turn 2: Interest rates depend on the type of account or product you're interested in. Are you looking for rates on a **savings account**, **CD (Certificate of Deposit)**, **loan**, or **mortgage**? Let me know so I can provide the most accurate information!


Notice the agent doesn't know the customer's name or that they asked about
savings accounts. Each call is independent.

## With a Session (Stateful)

Now let's create an `AgentSession` and pass it to every `agent.run()` call.
The session tracks the full conversation history.

In [11]:
# Create a session for this customer conversation
session = agent.create_session()

# Turn 1: Introduction
r1 = await agent.run(
    "Hi, my name is Priya and I'm interested in opening a savings account.",
    session=session,
)
print(f"Turn 1: {r1.text}\n")

Turn 1: Hi Priya, thank you for considering First National Bank! Our savings accounts are designed to help you grow your money with competitive interest rates and easy access. 

Could you tell me a bit about what you're looking for? For example, do you prefer no monthly fees, a higher interest rate, or other features like automatic transfers? I’ll recommend the best option for you!



In [12]:
# Turn 2: Follow-up about rates — agent remembers the savings account context
r2 = await agent.run(
    "What interest rate would I get?",
    session=session,
)
print(f"Turn 2: {r2.text}\n")

Turn 2: Our standard savings account currently offers an **annual percentage yield (APY) of 0.25%**, but we also offer a **high-yield savings account** with an APY of up to **2.50%**, depending on your balance. 

If you'd like, I can check the most recent rates for your area and help you compare options—does that sound good?



In [13]:
# Turn 3: Change topic to mortgage — agent still remembers the customer
r3 = await agent.run(
    "Actually, I'm also thinking about a mortgage. What are your current 30-year fixed rates?",
    session=session,
)
print(f"Turn 3: {r3.text}\n")

Turn 3: Great to hear you’re considering a mortgage, Priya! Our current **30-year fixed mortgage rates** start at around **7.15% APR**, depending on factors like your credit score, down payment, and loan amount. Rates can vary daily, so I recommend locking in a rate once you’re ready.

Would you like me to connect you with a mortgage specialist to discuss pre-approval or specific details?



In [14]:
# Turn 4: Reference something from earlier — agent connects the dots
r4 = await agent.run(
    "Between the savings account and the mortgage, which should I prioritize as a first-time customer?",
    session=session,
)
print(f"Turn 4: {r4.text}")

Turn 4: That depends on your financial goals, Priya! Here's a quick breakdown:

- If you're saving for a **down payment**, opening a high-yield savings account is a great way to build funds while earning interest.  
- If you're ready to **buy a home** soon and have enough saved, focusing on the mortgage process and pre-approval would be best.

If you're unsure, I recommend starting with the savings account to grow your funds, and we can explore mortgage options when you're ready. Let me know how you'd like to proceed!


The agent now remembers Priya's name, that she asked about savings
accounts and mortgages, and can give contextual advice.

## Serializing and Restoring Sessions

In a real app, you'd save the session when the customer logs off and
restore it when they come back. `AgentSession` supports this
via `to_dict()` and `from_dict()`.

In [15]:
from agent_framework import AgentSession

# Serialize the session (you'd store this in a database)
session_data = session.to_dict()
print(f"Session ID: {session_data.get('session_id', 'N/A')}")
print(f"Serialized keys: {list(session_data.keys())}")
print()

# Later, restore the session
restored_session = AgentSession.from_dict(session_data)

# Continue the conversation where we left off
r5 = await agent.run(
    "Remind me — what was my name again?",
    session=restored_session,
)
print(f"Restored turn: {r5.text}")

Session ID: bf9d6246-3ef6-4c9e-9101-f36eee1c019e
Serialized keys: ['type', 'session_id', 'service_session_id', 'state']

Restored turn: Your name is Priya! 😊 Let me know how else I can assist you, Priya!


In [16]:
await credential.close()
print("Done!")

Done!


---

## Key Takeaways

- Without a session, each `agent.run()` is stateless — no memory of prior turns
- `session = agent.create_session()` creates a conversation container
- Pass `session=session` to every `agent.run()` call to maintain continuity
- Serialize with `.to_dict()` → store in DB → restore with `AgentSession.from_dict()`
- Sessions are agent-specific — don't reuse across different agents

**Next up:** [05 — Agent as Tool](./05-agent-as-tool.ipynb) — compose agents
by using specialist agents as tools for an orchestrator.